# Sinh cache chuyển tự — Kanglish (chữ Latin) → chữ Kannada

**Chạy MỘT LẦN.** Kết quả là `data/xlit_kn.json`, commit vào repo; sau đó mọi máy chỉ đọc file
JSON — không cần cài IndicXlit, không cần GPU, không cần mạng.

**Settings (panel bên phải):** Accelerator = **GPU T4** · Internet = **On**

> **Vì sao phải chạy ở Kaggle:** IndicXlit phụ thuộc `fairseq`, mà `fairseq` **không build được
> trên Windows** — header của torch cần `/std:c++17` còn setup của fairseq không truyền cờ đó,
> nên dừng ở `error C2429: nested-namespace-definition`. Trên Linux thì cài bình thường.

In [ ]:
import os, subprocess, sys
REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"

TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    pass
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 1) Cài IndicXlit

Phải cài theo đúng thứ tự này, **không** cài thẳng `ai4bharat-transliteration`:

1. **`fairseq-fixed`** — bản vá của `fairseq` cho Python 3.11–3.12 (Kaggle đang dùng 3.12).
   `fairseq` gốc không build được ở đó.
2. **`--no-deps`** — `ai4bharat-transliteration` khai báo phụ thuộc `fairseq` **theo đúng tên
   đó**, nên pip sẽ cố cài bản gốc và chết, kể cả khi `fairseq-fixed` đã có. `--no-deps` cắt
   đường đó, đồng thời bỏ luôn `flask`, `gevent`, `tensorboardX` — thừa hoàn toàn với ta.
3. Cài tay đúng 4 gói nó thật sự cần lúc chạy.

`urduhack` cố tình **không** cài: nó chỉ dùng để chuẩn hoá chữ Shahmukhi (Urdu) nhưng lại kéo
theo **TensorFlow**, và còn hạ cấp `click` làm hỏng gói khác. `src/data/transliterate.py` đăng
ký sẵn một module giả mang tên đó trước khi import, nên nhánh Urdu không bao giờ chạy tới.

In [ ]:
!pip -q install ftfy
!pip -q install fairseq-fixed
!pip -q install --no-deps ai4bharat-transliteration
!pip -q install pydload indic-nlp-library ujson sacremoses

from src.data.transliterate import _stub_urduhack
_stub_urduhack()                                   # chan truoc khi import, tranh TensorFlow

from ai4bharat.transliteration import XlitEngine
e = XlitEngine("kn", beam_width=4, src_script_type="roman")
print(e.translit_sentence("nin sule maga"))        # mong doi: {'kn': 'ನಿನ್ ಸುಲೇ ಮಗ'}

## 2) Sinh cache

Gom mọi comment trong `data/raw` (4 file, **7.565 câu duy nhất** sau khi làm sạch), chuyển tự
từng câu rồi ghi ra `data/xlit_kn.json`.

Khoá của cache là **text đã làm sạch** — đúng chuỗi mà `preprocessing.py` đặt vào cột `text`,
nên lúc tra là khớp tuyệt đối.

Script **lưu sau mỗi 500 câu**: session bị ngắt thì chạy lại sẽ tiếp tục từ chỗ dừng.

In [ ]:
!python -m src.data.transliterate --lang kn --beam 4

## 3) Kiểm tra bằng mắt

Điều cần thấy: các **biến thể chính tả khác nhau của cùng một từ** phải cho ra **cùng một chuỗi
Kannada**. Đó chính là cơ chế làm giảm 25,6% OOV — nếu không thấy điều này thì chuyển tự sẽ
không giúp được gì.

In [ ]:
import json
from src.data.transliterate import CACHE

cache = json.load(open(CACHE, encoding='utf-8'))
print(f"{len(cache)} cau trong cache\n")

# cac bien the cua cung mot tu co ve cung mot chuoi Kannada khong?
probe = ["channel", "chanela", "channele", "chaannel", "aadre", "adre", "adru", "sule", "sulee"]
for w in probe:
    outs = {v.split()[i] for k, v in cache.items()
            for i, t in enumerate(k.lower().split()) if t == w and i < len(v.split())}
    print(f"  {w:10s} -> {sorted(outs)[:3] if outs else '(khong gap)'}")

print("\n--- vai cau day du ---")
for k, v in list(cache.items())[:5]:
    print(f"  {k[:55]:57s} -> {v[:55]}")

## 4) Tải về rồi commit

Tải `xlit_kn.json` từ panel **Output** bên phải, đặt vào `data/` ở máy, rồi:

```bash
git add data/xlit_kn.json
git commit -m "Add Roman->Kannada transliteration cache"
git push
```

Sau đó bật bằng `--set data.transliterate=true`. **Khi file test phát hành (20/9)**, chạy lại
notebook này một lần nữa — nó chỉ chuyển tự phần câu mới.

In [ ]:
import shutil
from src.data.transliterate import CACHE

shutil.copy(CACHE, "/kaggle/working/xlit_kn.json")
print(f"tai ve: /kaggle/working/xlit_kn.json  ({CACHE.stat().st_size/1e6:.1f} MB)")